# Ablation 6: TTA Minimal Diagnostic

**Purpose**: Check the Test-Time Augmentation assumptions in 2-3 minutes.

**Tests**:
1. Package installation (ultralytics, torch)
2. Baseline model access
3. TTA augmentation transforms (forward-inverse identity)
4. Single image TTA prediction
5. Ensemble aggregation

**Runtime**: 2-3 minutes (CPU only)
**Success**: all 5 tests pass. Next: smoke test.
**Failure**: any test fails. Record the exact error.

## Test 1: Package Installation

ultralytics and torch install correctly.

In [ ]:
# Install packages
!pip install -q 'ultralytics>=8.3.40' 'numpy<2.0' torch torchvision

# Validate installation
import importlib.util
assert importlib.util.find_spec('ultralytics') is not None, "ultralytics not installed"
assert importlib.util.find_spec('torch') is not None, "torch not installed"

import ultralytics
import torch
import numpy as np
print(f'Ultralytics: {ultralytics.__version__}')
print(f'PyTorch: {torch.__version__}')
print(f'NumPy: {np.__version__}')
print('\nOK TEST 1 PASSED: All packages installed successfully')

## Test 2: Baseline Model Access

Validate baseline model exists and loads correctly.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Check baseline model exists
model_path = Path('/kaggle/input/baseline-outputs/yolo_training/seed_42/weights/best.pt')
assert model_path.exists(), f"Baseline model not found: {model_path}"

# Load model
model = YOLO(str(model_path))
print(f'Model loaded: {type(model.model)}')
print(f'Model task: {model.task}')
assert model.task == 'segment', f"Expected segment model, got {model.task}"

print('\nOK TEST 2 PASSED: Baseline model loads successfully')

## Test 3: TTA Augmentation Transforms

Forward-inverse identity holds for all TTA augmentations.

**Check**: each augmentation is a perfect inverse (error < 1e-6).

In [ ]:
# Define TTA augmentations
AUGMENTATIONS = {
    "original": (
        lambda x: x,
        lambda x: x
    ),
    "horizontal_flip": (
        lambda x: torch.flip(x, dims=[-1]),
        lambda x: torch.flip(x, dims=[-1])
    ),
    "vertical_flip": (
        lambda x: torch.flip(x, dims=[-2]),
        lambda x: torch.flip(x, dims=[-2])
    ),
    "rotate_90": (
        lambda x: torch.rot90(x, k=1, dims=[-2, -1]),
        lambda x: torch.rot90(x, k=-1, dims=[-2, -1])
    ),
}

print(f'Testing {len(AUGMENTATIONS)} augmentations...')

# Test forward-inverse identity
test_tensor = torch.rand(3, 512, 512)
max_error = 0.0

for aug_name in AUGMENTATIONS:
    forward_fn, inverse_fn = AUGMENTATIONS[aug_name]
    transformed = forward_fn(test_tensor)
    reconstructed = inverse_fn(transformed)
    error = torch.abs(reconstructed - test_tensor).max().item()
    max_error = max(max_error, error)

    status = "OK" if error < 1e-6 else "FAIL"
    print(f'  {status} {aug_name}: error = {error:.2e}')

    if error >= 1e-6:
        raise AssertionError(f"{aug_name}: forward-inverse error too large: {error:.2e}")

print(f'\nMax error across all augmentations: {max_error:.2e}')
print('\nOK TEST 3 PASSED: All augmentation transforms are valid inverses')

## Test 4: Single Image TTA Prediction

Validate TTA prediction on a single test image.

In [ ]:
import cv2

# Load test image
test_img_dir = Path('/kaggle/input/baseline-outputs/data_wound_seg/test_images')
assert test_img_dir.exists(), f"Test images not found: {test_img_dir}"

test_images = list(test_img_dir.glob('*.png'))
assert len(test_images) > 0, "No test images found"

test_img_path = test_images[0]
print(f'Testing on: {test_img_path.name}')

# Load image
img = cv2.imread(str(test_img_path))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
print(f'Image shape: {img_rgb.shape}')

# Run baseline prediction
baseline_result = model(img_rgb, verbose=False)[0]
print(f'\nBaseline prediction:')
print(f'  Has masks: {baseline_result.masks is not None}')

if baseline_result.masks is not None:
    print(f'  Number of masks: {len(baseline_result.masks)}')

# Test TTA on each augmentation
print(f'\nTTA predictions:')
for aug_name in AUGMENTATIONS:
    forward_fn, inverse_fn = AUGMENTATIONS[aug_name]

    # Convert to tensor
    img_tensor = torch.from_numpy(img_rgb).permute(2, 0, 1).float()

    # Apply augmentation
    aug_img_tensor = forward_fn(img_tensor)
    aug_img = aug_img_tensor.permute(1, 2, 0).numpy().astype(np.uint8)

    # Predict
    result = model(aug_img, verbose=False)[0]

    print(f'  {aug_name}: masks={result.masks is not None}')

print('\nOK TEST 4 PASSED: TTA prediction works on all augmentations')

## Test 5: Ensemble Aggregation

Validate ensemble aggregation of TTA predictions.

In [ ]:
# Simulate multiple predictions for ensemble
# Note: Inline mask extraction (no external imports needed)
predictions = []

for aug_name in AUGMENTATIONS:
    forward_fn, inverse_fn = AUGMENTATIONS[aug_name]

    # Convert to tensor
    img_tensor = torch.from_numpy(img_rgb).permute(2, 0, 1).float()

    # Apply augmentation
    aug_img_tensor = forward_fn(img_tensor)
    aug_img = aug_img_tensor.permute(1, 2, 0).numpy().astype(np.uint8)

    # Predict
    result = model(aug_img, verbose=False)[0]

    # Extract mask (inline - no imports)
    if result.masks is not None and len(result.masks) > 0:
        mask_data = result.masks.data[0].cpu().numpy()

        # Resize to original size if needed
        if mask_data.shape != img_rgb.shape[:2]:
            mask_data = cv2.resize(mask_data, (img_rgb.shape[1], img_rgb.shape[0]))

        # Apply inverse transform
        mask_tensor = torch.from_numpy(mask_data).float()
        mask_reversed = inverse_fn(mask_tensor)
        predictions.append(mask_reversed.numpy())
    else:
        # No detection - add zero mask
        predictions.append(np.zeros(img_rgb.shape[:2], dtype=np.float32))

print(f'Collected {len(predictions)} predictions')

# Ensemble via mean aggregation
if len(predictions) > 0:
    ensemble_mask = (np.mean(predictions, axis=0) > 0.5).astype(np.uint8)
    print(f'Ensemble mask shape: {ensemble_mask.shape}')
    print(f'Ensemble mask pixels: {ensemble_mask.sum()} positive')
    print('\nOK TEST 5 PASSED: Ensemble aggregation works')
else:
    print('\nWARNING  TEST 5 WARNING: No predictions to ensemble (model detected nothing)')
    print('This is OK for diagnostic - model may not detect on all augmentations')
    print('\nOK TEST 5 PASSED: Ensemble logic works (even with zero predictions)')

## Test Summary

In [ ]:
import json
from datetime import datetime

test_summary = {
    'notebook': 'ablation_6_tta_minimal_diagnostic',
    'timestamp': datetime.now().isoformat(),
    'tests': [
        {'test': 1, 'name': 'Package Installation', 'status': 'Check output above'},
        {'test': 2, 'name': 'Baseline Model Access', 'status': 'Check output above'},
        {'test': 3, 'name': 'TTA Augmentation Transforms', 'status': 'Check output above'},
        {'test': 4, 'name': 'Single Image TTA Prediction', 'status': 'Check output above'},
        {'test': 5, 'name': 'Ensemble Aggregation', 'status': 'Check output above'}
    ],
    'augmentations_tested': list(AUGMENTATIONS.keys()),
    'runtime_target': '2-3 minutes',
    'next_step': 'If all tests pass → deploy smoke test (1 epoch + TTA)',
    'on_failure': 'Report exact error, fix root cause'
}

with open('/kaggle/working/test_summary.json', 'w') as f:
    json.dump(test_summary, f, indent=2)

print('\n' + '='*60)
print('MINIMAL DIAGNOSTIC COMPLETE')
print('='*60)
print('\nReview test results above:')
print('  - All tests should show OK PASSED')
print('  - Any FAIL FAILED indicates assumption violation')
print('\nNext steps:')
print('  - If ALL pass: Deploy smoke test (1 epoch training + TTA)')
print('  - If ANY fail: Debug root cause before proceeding')
print('\nOutputs:')
print('  - test_summary.json')